# Grape Leaf Disease Classification and Severity Analysis

This notebook provides a comprehensive pipeline for:
1. **Classification** of grape leaf diseases (Healthy, Downy Mildew, Bacterial Leaf Spot, Powdery Mildew).
2. **Comparison** between baseline models and an Ensemble model.
3. **Segmentation** of diseased areas to calculate infection severity.
4. **Visualization** of intermediate results.

### Instructions:
- If using **Google Colab**, upload your dataset to Google Drive and update `DATASET_PATH`.
- Ensure your dataset is organized into folders: `Healthy`, `Bacterial_Leaf_Spot`, `Downy_Mildew`, `Powdery_Mildew`.

In [ ]:
# Install dependencies if needed
# !pip install torch torchvision torchaudio scikit-learn matplotlib opencv-python pillow

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import cv2
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from PIL import Image
import copy

# Optional: Mount Google Drive if using Colab
# from google.colab import drive
# drive.mount('/content/drive')

## 1. Data Preparation and Splitting

In [ ]:
# Dataset Path Configuration
DATASET_PATH = 'path_to_your_dataset' # Update this path

def get_data_loaders(data_dir, batch_size=32, train_split=0.7, val_split=0.15, test_split=0.15):
    transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    full_dataset = datasets.ImageFolder(data_dir, transform=transform)
    
    # Get indices for split
    train_idx, temp_idx = train_test_split(list(range(len(full_dataset))), train_size=train_split, stratify=full_dataset.targets)
    val_idx, test_idx = train_test_split(temp_idx, train_size=val_split/(val_split + test_split), stratify=[full_dataset.targets[i] for i in temp_idx])

    train_data = Subset(full_dataset, train_idx)
    val_data = Subset(full_dataset, val_idx)
    test_data = Subset(full_dataset, test_idx)

    loaders = {
        'train': DataLoader(train_data, batch_size=batch_size, shuffle=True),
        'val': DataLoader(val_data, batch_size=batch_size, shuffle=False),
        'test': DataLoader(test_data, batch_size=batch_size, shuffle=False)
    }
    
    return loaders, full_dataset.classes

# loaders, class_names = get_data_loaders(DATASET_PATH)

## 2. Model Architectures

We define two baseline models (ResNet18 and MobileNetV2) and an Ensemble model.

In [ ]:
def get_baseline_resnet18(num_classes):
    # Using modern weights parameter
    weights = models.ResNet18_Weights.DEFAULT
    model = models.resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

def get_baseline_mobilenet(num_classes):
    weights = models.MobileNet_V2_Weights.DEFAULT
    model = models.mobilenet_v2(weights=weights)
    model.classifier[1] = nn.Linear(model.last_channel, num_classes)
    return model

class EnsembleModel(nn.Module):
    def __init__(self, modelA, modelB, num_classes):
        super(EnsembleModel, self).__init__()
        self.modelA = modelA
        self.modelB = modelB
        # Remove classifiers
        self.modelA.fc = nn.Identity()
        self.modelB.classifier = nn.Identity()
        
        self.classifier = nn.Sequential(
            nn.Linear(512 + 1280, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x1 = self.modelA(x)
        x2 = self.modelB(x)
        x = torch.cat((x1, x2), dim=1)
        x = self.classifier(x)
        return x

## 3. Training Logic

In [ ]:
def train_model(model, loaders, criterion, optimizer, num_epochs=10, device='cuda'):
    model = model.to(device)
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in loaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / len(loaders[phase].dataset)
            epoch_acc = running_corrects.double() / len(loaders[phase].dataset)

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
            
            if phase == 'train':
                history['train_loss'].append(epoch_loss)
                history['train_acc'].append(epoch_acc.item())
            else:
                history['val_loss'].append(epoch_loss)
                history['val_acc'].append(epoch_acc.item())

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

    print(f'Best val Acc: {best_acc:4f}')
    model.load_state_dict(best_model_wts)
    return model, history

def evaluate_model(model, loader, class_names, device='cuda'):
    model.eval()
    y_true = []
    y_pred = []
    y_score = []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())
            y_score.extend(torch.softmax(outputs, dim=1).cpu().numpy())

    return np.array(y_true), np.array(y_pred), np.array(y_score)

## 5. ROC Plotting and Comparison

In [ ]:
def plot_roc_comparison(model_results, class_names):
    # model_results: dict mapping model_name to (y_true, y_score)
    plt.figure(figsize=(10, 8))
    
    for model_name, (y_true, y_score) in model_results.items():
        # Compute ROC for each class and average
        fpr = dict()
        tpr = dict()
        roc_auc = dict()
        
        n_classes = len(class_names)
        for i in range(n_classes):
            y_true_binary = (y_true == i).astype(int)
            fpr[i], tpr[i], _ = roc_curve(y_true_binary, y_score[:, i])
            roc_auc[i] = auc(fpr[i], tpr[i])
            
        # Compute micro-average ROC curve and ROC area
        fpr_micro, tpr_micro, _ = roc_curve(np.eye(n_classes)[y_true].ravel(), y_score.ravel())
        roc_auc_micro = auc(fpr_micro, tpr_micro)
        
        plt.plot(fpr_micro, tpr_micro, label=f'{model_name} (AUC = {roc_auc_micro:0.2f})')

    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (Micro-average Comparison)')
    plt.legend(loc="lower right")
    plt.show()

## 6. Segmentation and Severity Calculation

This module handles background removal and lesion segmentation using color-space analysis.

In [ ]:
def segment_leaf(image_path):
    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Convert to HSV for better color segmentation
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    
    # Typical green color range for leaves
    lower_green = np.array([25, 40, 40])
    upper_green = np.array([90, 255, 255])
    
    mask = cv2.inRange(hsv, lower_green, upper_green)
    
    # Refine mask with morphological operations
    kernel = np.ones((5,5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    
    leaf_only = cv2.bitwise_and(image_rgb, image_rgb, mask=mask)
    
    return image_rgb, mask, leaf_only

def segment_diseased_areas(leaf_rgb, leaf_mask):
    # Convert leaf to Lab color space to isolate lesions
    lab = cv2.cvtColor(leaf_rgb, cv2.COLOR_RGB2Lab)
    l, a, b = cv2.split(lab)
    
    # Healthy leaf is usually green (low 'a' value in Lab color space)
    # Lesions (Downy Mildew, spots, etc.) are usually brown, yellow, or grey
    # which results in higher 'a' or 'b' values.
    # We use Otsu's thresholding for automatic lesion detection
    # Focus on the 'a' channel (green-red) or 'b' channel (blue-yellow)
    
    # Applying a blur to reduce noise
    a_blurred = cv2.GaussianBlur(a, (5, 5), 0)
    
    # Use Otsu thresholding to find the best cut-off
    _, disease_mask = cv2.threshold(a_blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # Ensure disease mask is only within the leaf
    disease_mask = cv2.bitwise_and(disease_mask, disease_mask, mask=leaf_mask)
    
    return disease_mask

def calculate_severity(leaf_mask, disease_mask):
    leaf_area = np.sum(leaf_mask > 0)
    disease_area = np.sum(disease_mask > 0)
    
    if leaf_area == 0: return 0.0
    
    severity = (disease_area / leaf_area) * 100
    return severity

## 7. Training and Evaluation Experiment

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loaders, class_names = get_data_loaders(DATASET_PATH, batch_size=32)
num_classes = len(class_names)

# 1. Baseline: ResNet18
print("Training ResNet18 Baseline...")
resnet18 = get_baseline_resnet18(num_classes)
optimizer_resnet = optim.AdamW(resnet18.parameters(), lr=0.001)
resnet18, _ = train_model(resnet18, loaders, nn.CrossEntropyLoss(), optimizer_resnet, num_epochs=10, device=device)

# 2. Baseline: MobileNetV2
print("\nTraining MobileNetV2 Baseline...")
mobilenet = get_baseline_mobilenet(num_classes)
optimizer_mobile = optim.AdamW(mobilenet.parameters(), lr=0.001)
mobilenet, _ = train_model(mobilenet, loaders, nn.CrossEntropyLoss(), optimizer_mobile, num_epochs=10, device=device)

# 3. Proposed Ensemble Model
print("\nTraining Ensemble Model...")
ensemble = EnsembleModel(get_baseline_resnet18(num_classes), get_baseline_mobilenet(num_classes), num_classes)
optimizer_ensemble = optim.AdamW(ensemble.parameters(), lr=0.001)
ensemble, _ = train_model(ensemble, loaders, nn.CrossEntropyLoss(), optimizer_ensemble, num_epochs=10, device=device)

# Evaluation and ROC Comparison
results = {}
for name, model in [('ResNet18', resnet18), ('MobileNetV2', mobilenet), ('Ensemble', ensemble)]:
    y_true, y_pred, y_score = evaluate_model(model, loaders['test'], class_names, device=device)
    results[name] = (y_true, y_score)
    print(f"\n{name} Classification Report:")
    print(classification_report(y_true, y_pred, target_names=class_names))

plot_roc_comparison(results, class_names)

## 8. Full Pipeline and Visualization

In [ ]:
def process_and_visualize(image_path, model, class_names, device='cuda'):
    # 1. Classification
    model.eval()
    img_pil = Image.open(image_path).convert('RGB')
    preprocess = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    img_tensor = preprocess(img_pil).unsqueeze(0).to(device)
    
    with torch.no_grad():
        outputs = model(img_tensor)
        _, preds = torch.max(outputs, 1)
        predicted_class = class_names[preds[0]]
    
    # 2. Segmentation
    orig_rgb, leaf_mask, leaf_only = segment_leaf(image_path)
    disease_mask = segment_diseased_areas(orig_rgb, leaf_mask)
    severity = calculate_severity(leaf_mask, disease_mask)
    
    # 3. Visualization
    plt.figure(figsize=(24, 6))
    
    plt.subplot(1, 5, 1)
    plt.imshow(orig_rgb)
    plt.title("1. Original Image")
    
    plt.subplot(1, 5, 2)
    plt.imshow(leaf_mask, cmap='gray')
    plt.title("2. Preprocessing (Mask)")
    
    plt.subplot(1, 5, 3)
    plt.imshow(leaf_only)
    plt.title("3. Background Removed")
    
    plt.subplot(1, 5, 4)
    plt.imshow(disease_mask, cmap='hot')
    plt.title("4. Disease Segmentation")
    
    plt.subplot(1, 5, 5)
    # Overlay disease mask on original leaf for better visualization
    overlay = leaf_only.copy()
    overlay[disease_mask > 0] = [255, 0, 0] # Highlight disease in red
    plt.imshow(overlay)
    plt.title(f"5. Final: {predicted_class}\nSeverity: {severity:.2f}%")
    
    plt.tight_layout()
    plt.show()